# Datathon 2026 Task 1 - Graph and guarded-text blend inference

Self-contained inference for candidate `d1-e010-graphtextblend`. It fits the deterministic official-graph model and three-sigma guarded event-text model, then averages their predictions 50:50. It uses only official competition inputs, NumPy, and pandas; no API, credentials, external data, or pretrained weights.

In [ ]:
import hashlib
import json
import os
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

HISTORY_LENGTH = 15
HORIZONS = np.array([5, 10, 15], dtype=np.int64)
ROAD_COUNT = 1260
ALPHA = 0.1
RESIDUAL_ALPHA = 1.0
Z_THRESHOLD = 3.0
CHUNK_SIZE = 256
started = time.perf_counter()

def is_data_root(path):
    path = Path(path)
    return (
        (path / 'train').is_dir()
        and (path / 'test' / 'test_X_hist.npy').is_file()
        and (path / 'sample_submission.csv').is_file()
    )

override = os.environ.get('DATATHON_DATA_ROOT')
kaggle_mode = False
if override:
    data_root = Path(override)
    if not is_data_root(data_root):
        raise FileNotFoundError(f'DATATHON_DATA_ROOT is not a valid dataset root: {data_root}')
else:
    kaggle_input = Path('/kaggle/input')
    kaggle_candidates = sorted({
        path.parent
        for path in kaggle_input.rglob('sample_submission.csv')
        if is_data_root(path.parent)
    }) if kaggle_input.is_dir() else []
    if len(kaggle_candidates) > 1:
        raise FileNotFoundError(f'Found multiple Task 1 datasets under /kaggle/input: {kaggle_candidates}')
    if kaggle_candidates:
        data_root = kaggle_candidates[0]
        kaggle_mode = True
    else:
        local_candidates = set()
        for base in (Path.cwd(), *Path.cwd().parents):
            for relative in (
                Path('task1/data/competition/dataset-task1'),
                Path('data/competition/dataset-task1'),
            ):
                candidate = base / relative
                if is_data_root(candidate):
                    local_candidates.add(candidate.resolve())
        candidates = sorted(local_candidates)
        if len(candidates) != 1:
            raise FileNotFoundError(
                'Could not identify one Task 1 dataset. Run from the repository or set DATATHON_DATA_ROOT. '
                f'Found: {candidates}'
            )
        data_root = candidates[0]

output_override = os.environ.get('DATATHON_OUTPUT_PATH')
if output_override:
    output_path = Path(output_override)
elif kaggle_mode:
    output_path = Path('/kaggle/working/submission.csv')
else:
    task1_root = next((path for path in (data_root, *data_root.parents) if path.name == 'task1'), None)
    if task1_root is None:
        raise FileNotFoundError('Could not identify the local task1 directory; set DATATHON_OUTPUT_PATH')
    output_path = task1_root / 'submissions' / 'submission_d1-e010-graphtextblend_local.csv'
print(f'data_root={data_root}')
print(f'output_path={output_path}')


In [ ]:
def windows_at_origins(block, origins):
    origins = np.asarray(origins, dtype=np.int64)
    offsets = np.arange(HISTORY_LENGTH - 1, -1, -1, dtype=np.int64)
    histories = np.asarray(block[origins[:, None] - offsets[None, :]], dtype=np.float32)
    targets = np.asarray(block[origins[:, None] + HORIZONS[None, :]], dtype=np.float32)
    return histories, targets

def history_features(history):
    history = np.asarray(history, dtype=np.float32)
    if history.ndim != 3 or history.shape[1] != HISTORY_LENGTH:
        raise ValueError(f'Expected (samples, 15, roads), got {history.shape}')
    recent = history[:, -5:]
    x = np.arange(5, dtype=np.float32)
    centered = x - x.mean()
    slope5 = np.einsum('str,t->sr', recent, centered) / float(np.square(centered).sum())
    return np.stack((
        history[:, -1],
        history[:, -3:].mean(axis=1),
        recent.mean(axis=1),
        history.mean(axis=1),
        slope5,
    ), axis=2).astype(np.float32, copy=False)

@dataclass
class RoadRidgeModel:
    feature_mean: np.ndarray
    feature_scale: np.ndarray
    target_mean: np.ndarray
    coefficients: np.ndarray

    def predict(self, history):
        features = history_features(history).astype(np.float64)
        standardized = (features - self.feature_mean[None, :, :]) / self.feature_scale[None, :, :]
        prediction = self.target_mean[None, :, :] + np.einsum('nrf,rfh->nrh', standardized, self.coefficients)
        prediction = prediction.transpose(0, 2, 1)
        zero_history = np.all(np.asarray(history) == 0, axis=1)
        prediction = np.where(zero_history[:, None, :], 0.0, prediction)
        return np.maximum(prediction, 0.0).astype(np.float32)

def fit_road_ridge(block, origin_start, origin_end, alpha=ALPHA, chunk_size=CHUNK_SIZE):
    road_count = block.shape[1]
    feature_count = 5
    horizon_count = len(HORIZONS)
    sum_x = np.zeros((road_count, feature_count), dtype=np.float64)
    sum_y = np.zeros((road_count, horizon_count), dtype=np.float64)
    sum_xx = np.zeros((road_count, feature_count, feature_count), dtype=np.float64)
    sum_xy = np.zeros((road_count, feature_count, horizon_count), dtype=np.float64)
    sample_count = 0
    for start in range(origin_start, origin_end + 1, chunk_size):
        stop = min(start + chunk_size, origin_end + 1)
        histories, targets = windows_at_origins(block, np.arange(start, stop))
        x = history_features(histories).astype(np.float64)
        y = targets.transpose(0, 2, 1).astype(np.float64)
        sum_x += x.sum(axis=0)
        sum_y += y.sum(axis=0)
        sum_xx += np.einsum('nrf,nrg->rfg', x, x, optimize=True)
        sum_xy += np.einsum('nrf,nrh->rfh', x, y, optimize=True)
        sample_count += len(x)
    feature_mean = sum_x / sample_count
    target_mean = sum_y / sample_count
    covariance = sum_xx / sample_count - np.einsum('rf,rg->rfg', feature_mean, feature_mean)
    cross_covariance = sum_xy / sample_count - np.einsum('rf,rh->rfh', feature_mean, target_mean)
    variance = np.maximum(np.diagonal(covariance, axis1=1, axis2=2), 0.0)
    feature_scale = np.sqrt(variance)
    feature_scale = np.where(feature_scale > 1e-6, feature_scale, 1.0)
    standardized_covariance = covariance / (feature_scale[:, :, None] * feature_scale[:, None, :])
    standardized_cross_covariance = cross_covariance / feature_scale[:, :, None]
    system = standardized_covariance + alpha * np.eye(feature_count)[None, :, :]
    coefficients = np.linalg.solve(system, standardized_cross_covariance)
    return RoadRidgeModel(feature_mean, feature_scale, target_mean, coefficients)

def build_neighbor_edges(adjacency):
    adjacency = np.asarray(adjacency)
    if adjacency.ndim != 2 or adjacency.shape[0] != adjacency.shape[1]:
        raise ValueError(f'Expected square adjacency, got {adjacency.shape}')
    if not np.isfinite(adjacency).all():
        raise ValueError('adjacency must be finite')
    connected = (adjacency != 0) | (adjacency.T != 0)
    np.fill_diagonal(connected, False)
    rows, columns = np.nonzero(connected)
    degree = np.bincount(rows, minlength=len(adjacency))
    weights = 1.0 / degree[rows]
    return rows.astype(np.int64), columns.astype(np.int64), weights.astype(np.float32)

def graph_history_features(history, edge_rows, edge_columns, edge_weights, edge_chunk_size=1024):
    local = history_features(history)
    neighbor = np.zeros_like(local)
    for start in range(0, len(edge_rows), edge_chunk_size):
        stop = min(start + edge_chunk_size, len(edge_rows))
        rows = edge_rows[start:stop]
        columns = edge_columns[start:stop]
        contribution = local[:, columns, :] * edge_weights[None, start:stop, None]
        np.add.at(neighbor, (slice(None), rows, slice(None)), contribution)
    return np.concatenate((local, neighbor), axis=2).astype(np.float32, copy=False)

@dataclass
class GraphRidgeModel:
    feature_mean: np.ndarray
    feature_scale: np.ndarray
    target_mean: np.ndarray
    coefficients: np.ndarray
    edge_rows: np.ndarray
    edge_columns: np.ndarray
    edge_weights: np.ndarray

    def predict(self, history):
        features = graph_history_features(history, self.edge_rows, self.edge_columns, self.edge_weights).astype(np.float64)
        standardized = (features - self.feature_mean[None, :, :]) / self.feature_scale[None, :, :]
        prediction = self.target_mean[None, :, :] + np.einsum('nrf,rfh->nrh', standardized, self.coefficients, optimize=True)
        prediction = prediction.transpose(0, 2, 1)
        zero_history = np.all(np.asarray(history) == 0, axis=1)
        prediction = np.where(zero_history[:, None, :], 0.0, prediction)
        return np.maximum(prediction, 0.0).astype(np.float32)

def fit_graph_ridge(block, adjacency, origin_start, origin_end, alpha=ALPHA, chunk_size=CHUNK_SIZE):
    edge_rows, edge_columns, edge_weights = build_neighbor_edges(adjacency)
    road_count = block.shape[1]
    feature_count = 10
    horizon_count = len(HORIZONS)
    sum_x = np.zeros((road_count, feature_count), dtype=np.float64)
    sum_y = np.zeros((road_count, horizon_count), dtype=np.float64)
    sum_xx = np.zeros((road_count, feature_count, feature_count), dtype=np.float64)
    sum_xy = np.zeros((road_count, feature_count, horizon_count), dtype=np.float64)
    sample_count = 0
    for start in range(origin_start, origin_end + 1, chunk_size):
        stop = min(start + chunk_size, origin_end + 1)
        histories, targets = windows_at_origins(block, np.arange(start, stop))
        x = graph_history_features(histories, edge_rows, edge_columns, edge_weights).astype(np.float64)
        y = targets.transpose(0, 2, 1).astype(np.float64)
        sum_x += x.sum(axis=0)
        sum_y += y.sum(axis=0)
        sum_xx += np.einsum('nrf,nrg->rfg', x, x, optimize=True)
        sum_xy += np.einsum('nrf,nrh->rfh', x, y, optimize=True)
        sample_count += len(x)
    feature_mean = sum_x / sample_count
    target_mean = sum_y / sample_count
    covariance = sum_xx / sample_count - np.einsum('rf,rg->rfg', feature_mean, feature_mean)
    cross_covariance = sum_xy / sample_count - np.einsum('rf,rh->rfh', feature_mean, target_mean)
    variance = np.maximum(np.diagonal(covariance, axis1=1, axis2=2), 0.0)
    feature_scale = np.sqrt(variance)
    feature_scale = np.where(feature_scale > 1e-6, feature_scale, 1.0)
    standardized_covariance = covariance / (feature_scale[:, :, None] * feature_scale[:, None, :])
    standardized_cross_covariance = cross_covariance / feature_scale[:, :, None]
    system = standardized_covariance + alpha * np.eye(feature_count)[None, :, :]
    coefficients = np.linalg.solve(system, standardized_cross_covariance)
    return GraphRidgeModel(feature_mean, feature_scale, target_mean, coefficients, edge_rows, edge_columns, edge_weights)

EVENT_PATTERNS = (
    'a general traffic accident', 'road closure', 'construction',
    'road traffic control', 'an announcement', 'prohibit left turn',
)
GUARDED_FEATURE_INDEX = EVENT_PATTERNS.index('prohibit left turn')

def text_features(texts):
    rows = []
    for text in texts:
        if not isinstance(text, str):
            raise TypeError('all text entries must be strings')
        lowered = text.lower()
        counts = [lowered.count(pattern) for pattern in EVENT_PATTERNS]
        total_events = sum(bool(part.strip()) for part in lowered.split('.'))
        rows.append((*counts, total_events))
    return np.asarray(rows, dtype=np.float64)

def load_aligned_texts(path, expected_keys):
    data = json.loads(Path(path).read_text(encoding='utf-8'))
    expected_keys = list(expected_keys)
    if not isinstance(data, dict) or list(data) != expected_keys:
        raise ValueError(f'Text keys in {path} do not match expected alignment')
    values = list(data.values())
    if not all(isinstance(value, str) for value in values):
        raise ValueError(f'Text values in {path} must all be strings')
    return values

@dataclass
class TextZGuardModel:
    base_model: RoadRidgeModel
    feature_mean: np.ndarray
    feature_scale: np.ndarray
    residual_mean: np.ndarray
    coefficients: np.ndarray

    def predict(self, history, texts):
        base = self.base_model.predict(history).astype(np.float64)
        standardized = (text_features(texts) - self.feature_mean) / self.feature_scale
        mask = np.abs(standardized[:, GUARDED_FEATURE_INDEX]) > Z_THRESHOLD
        standardized[mask, GUARDED_FEATURE_INDEX] = 0.0
        residual = self.residual_mean[None, :, :] + np.einsum('nf,rfh->nrh', standardized, self.coefficients, optimize=True)
        prediction = base + residual.transpose(0, 2, 1)
        zero_history = np.all(np.asarray(history) == 0, axis=1)
        prediction = np.where(zero_history[:, None, :], 0.0, prediction)
        return np.maximum(prediction, 0.0).astype(np.float32)

def fit_text_zguard(block, texts, origin_start, origin_end, chunk_size=CHUNK_SIZE):
    base_model = fit_road_ridge(block, origin_start, origin_end, alpha=ALPHA, chunk_size=chunk_size)
    origins = np.arange(origin_start, origin_end + 1, dtype=np.int64)
    raw_features = text_features([texts[index] for index in origins])
    feature_mean = raw_features.mean(axis=0)
    feature_scale = raw_features.std(axis=0)
    feature_scale = np.where(feature_scale > 1e-6, feature_scale, 1.0)
    standardized = (raw_features - feature_mean) / feature_scale
    road_count = block.shape[1]
    feature_count = standardized.shape[1]
    horizon_count = len(HORIZONS)
    sum_residual = np.zeros((road_count, horizon_count), dtype=np.float64)
    sum_cross = np.zeros((feature_count, road_count, horizon_count), dtype=np.float64)
    for start in range(0, len(origins), chunk_size):
        chunk_origins = origins[start:start + chunk_size]
        histories, targets = windows_at_origins(block, chunk_origins)
        base_prediction = base_model.predict(histories)
        residual = (targets.astype(np.float64) - base_prediction.astype(np.float64)).transpose(0, 2, 1)
        x = standardized[start:start + len(chunk_origins)]
        sum_residual += residual.sum(axis=0)
        sum_cross += np.einsum('nf,nrh->frh', x, residual, optimize=True)
    sample_count = len(origins)
    residual_mean = sum_residual / sample_count
    covariance = standardized.T @ standardized / sample_count
    cross_covariance = sum_cross / sample_count
    system = covariance + RESIDUAL_ALPHA * np.eye(feature_count)
    solved = np.linalg.solve(system, cross_covariance.reshape(feature_count, -1))
    coefficients = solved.reshape(feature_count, road_count, horizon_count).transpose(1, 0, 2)
    return TextZGuardModel(base_model, feature_mean, feature_scale, residual_mean, coefficients)

def classify_test_regime(history, zero_road_threshold=100):
    zero_road_count = np.all(np.asarray(history) == 0, axis=1).sum(axis=1)
    return (zero_road_count > zero_road_threshold).astype(np.int64)


In [ ]:
train_paths = sorted((data_root / 'train').glob('train_speed_*.npy'))
if len(train_paths) != 2:
    raise ValueError(f'Expected two train speed blocks, found {train_paths}')
blocks = [np.load(path, mmap_mode='r') for path in train_paths]
text_paths = sorted((data_root / 'train').glob('train_text_*.json'))
if len(text_paths) != 2:
    raise ValueError(f'Expected two train text blocks, found {text_paths}')
text_blocks = []
for index, (path, block) in enumerate(zip(text_paths, blocks, strict=True), start=1):
    keys = [f'm{index}_{position}' for position in range(1, len(block) + 1)]
    text_blocks.append(load_aligned_texts(path, keys))
adjacency = np.load(data_root / 'static' / 'matrix.npy')
test_history = np.load(data_root / 'test' / 'test_X_hist.npy', mmap_mode='r')
if test_history.shape != (540, 15, ROAD_COUNT):
    raise ValueError(f'Unexpected test shape: {test_history.shape}')
test_keys = [f'test_{index:05d}' for index in range(len(test_history))]
test_texts = np.asarray(load_aligned_texts(data_root / 'test' / 'test_texts.json', test_keys), dtype=object)
graph_models = [
    fit_graph_ridge(block, adjacency, 14, len(block) - int(HORIZONS.max()) - 1)
    for block in blocks
]
text_models = [
    fit_text_zguard(block, texts, 14, len(block) - int(HORIZONS.max()) - 1)
    for block, texts in zip(blocks, text_blocks, strict=True)
]
regimes = classify_test_regime(test_history)
if [int((regimes == index).sum()) for index in range(2)] != [372, 168]:
    raise ValueError('Unexpected test regime counts')
graph_prediction = np.empty((len(test_history), len(HORIZONS), ROAD_COUNT), dtype=np.float32)
text_prediction = np.empty_like(graph_prediction)
for regime_index, (graph_model, text_model) in enumerate(zip(graph_models, text_models, strict=True)):
    mask = regimes == regime_index
    graph_prediction[mask] = graph_model.predict(test_history[mask])
    text_prediction[mask] = text_model.predict(test_history[mask], test_texts[mask].tolist())
prediction = (0.5 * graph_prediction.astype(np.float64) + 0.5 * text_prediction.astype(np.float64)).astype(np.float32)
if not np.isfinite(prediction).all() or (prediction < 0).any():
    raise ValueError('Predictions must be finite and nonnegative')
zero_history = np.all(test_history == 0, axis=1)
if not (prediction.transpose(0, 2, 1)[zero_history] == 0).all():
    raise ValueError('Zero-history guard failed')


In [ ]:
template = pd.read_csv(data_root / 'sample_submission.csv')
if template.columns.tolist() != ['id', 'speed']:
    raise ValueError(f'Unexpected template columns: {template.columns.tolist()}')
expected_rows = prediction.size
if len(template) != expected_rows:
    raise ValueError(f'Template has {len(template)} rows, expected {expected_rows}')
expected_ids = [
    f'test_{sample:05d}_h{int(horizon)}_r{road}'
    for sample in range(prediction.shape[0])
    for horizon in HORIZONS
    for road in range(prediction.shape[2])
]
if template['id'].tolist() != expected_ids:
    raise ValueError('Template ID order does not match sample-horizon-road order')
template['speed'] = prediction.reshape(-1)
output_path.parent.mkdir(parents=True, exist_ok=True)
template.to_csv(output_path, index=False)
summary = {
    'experiment_id': 'd1-e010-graphtextblend',
    'blend_weights': {'graphres': 0.5, 'textzguard': 0.5},
    'rows': int(prediction.size),
    'regime_counts': [int((regimes == index).sum()) for index in range(2)],
    'zero_history_pairs': int(zero_history.sum()),
    'min': float(prediction.min()),
    'max': float(prediction.max()),
    'mean': float(prediction.mean()),
    'float32_prediction_sha256': hashlib.sha256(prediction.astype('<f4', copy=False).tobytes()).hexdigest(),
    'runtime_seconds': float(time.perf_counter() - started),
    'output': str(output_path),
}
print(json.dumps(summary, indent=2))
